Multilingual Characterization and Extraction of Narratives
from Online News - Subtask 1
--

Author: Manuel Carlucci

Student ID: 855237

Email: m.carlucci69@studenti.uniba.it

 Subtask 2: Narrative Classification
 ---

Definition: given a news article and a two-level taxonomy of narrative labels (where each narrative is subdivided into subnarratives) from a particular
domain, assign to the article all the appropriate subnarrative labels. This is a multi-label multi-class document classification task

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# - paths -
BASE = '/content/drive/MyDrive/NLP_Project'

NARRATIVES        = f'{BASE}/subtask_2/subtask2_narratives.txt'
SUBNARRATIVES     = f'{BASE}/subtask_2/subtask2_subnarratives.txt'

TRAIN_DOCS_DIR    = f'{BASE}/training/EN/raw-documents'
TRAIN_ANNOTATIONS = f'{BASE}/subtask_2/training/EN/subtask-2-annotations.txt'
TEST_DOCS_DIR     = f'{BASE}/subtask_2/test/EN/subtask-2-documents'
GOLD_FILE         = f'{BASE}/subtask_2/test/EN/subtask-2-annotations.txt'
OUTPUT_FILE       = f'{BASE}/predictions_subtask2_EN.txt'

LANG = 'EN'  # only English

Mounted at /content/drive


In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer


Load narrative / sub-narrative label lists

In [ ]:
def load_label_list(path):
    """Read a plain-text label file (one label per line)."""
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

narrative_labels    = load_label_list(NARRATIVES)
subnarrative_labels = load_label_list(SUBNARRATIVES)

print(f"Dominant narrative labels  : {len(narrative_labels)}")
print(f"Sub-narrative labels       : {len(subnarrative_labels)}")

Dominant narrative labels  : 22
Sub-narrative labels       : 91


Now we parse the raw annotation file into a structured dataframe, extracting article IDs, language information, and multi-label dominant and sub-narrative annotations. Labels are cleaned and deduplicated while preserving order, and the dataset is filtered to retain only English instances for subsequent processing.

In [ ]:
def parse_annotations(annotation_file):
    records = []
    with open(annotation_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            article_id = parts[0].strip()
            lang = article_id.split('_')[0] if '_' in article_id else 'UNK'

            dom_str = parts[1].strip() if len(parts) > 1 else 'Other'
            sub_str = parts[2].strip() if len(parts) > 2 else dom_str

            # deduplicate while preserving order
            dom_labels = list(dict.fromkeys(l.strip() for l in dom_str.split(';') if l.strip()))
            sub_labels = list(dict.fromkeys(l.strip() for l in sub_str.split(';') if l.strip()))

            records.append({
                'article_id'    : article_id,
                'language'      : lang,
                'dom_narratives': dom_labels,
                'subnarratives' : sub_labels,
            })
    return pd.DataFrame(records)

df_raw = parse_annotations(TRAIN_ANNOTATIONS)
# Keep only EN
df_raw = df_raw[df_raw['language'] == LANG].reset_index(drop=True)
print(f"EN training instances: {len(df_raw)}")
df_raw.head(3)

EN training instances: 399


,article_id,language,dom_narratives,subnarratives
0,EN_CC_100013.txt,EN,[CC: Criticism of climate movement],[CC: Criticism of climate movement: Ad hominem...
1,EN_UA_300009.txt,EN,[Other],[Other]
2,EN_UA_300017.txt,EN,[Other],[Other]


Let's load the raw article texts from disk and aligns them with the annotation dataframe. For each article ID, it searches for filename matching the ID and builds a dictionary mapping IDs to texts. The texts are then merged into df_raw, and a quick check is performed to verify how many articles were successfully loaded versus those with missing or empty content.

In [ ]:
def load_texts(docs_dir, article_ids):
    """Read raw text files; return a dict {article_id: text}."""
    texts = {}
    for aid in article_ids:
        # Files may be named <article_id>.txt or just <article_id>
        for fname in [f'{aid}.txt', str(aid)]:
            fpath = os.path.join(docs_dir, fname)
            if os.path.exists(fpath):
                with open(fpath, 'r', encoding='utf-8', errors='replace') as f:
                    texts[aid] = f.read()
                break
        else:
            texts[aid] = ''
    return texts

texts = load_texts(TRAIN_DOCS_DIR, df_raw['article_id'].tolist())
df_raw['text'] = df_raw['article_id'].map(texts)
df_raw.head(5)
print(f"Texts loaded: {df_raw['text'].str.len().gt(0).sum()} / {len(df_raw)}")
print(f"df_raw['text'].str.len(): {df_raw['text'].str.len()}")
print(f"df_raw['text'].str.len().gt(0): {df_raw['text'].str.len().gt(0)}")

Texts loaded: 399 / 399
df_raw['text'].str.len(): 0      2021
1      2106
2      1558
3      2922
4      2676
       ... 
394    2487
395    2958
396    2219
397    2045
398    2981
Name: text, Length: 399, dtype: int64
df_raw['text'].str.len().gt(0): 0      True
1      True
2      True
3      True
4      True
       ... 
394    True
395    True
396    True
397    True
398    True
Name: text, Length: 399, dtype: bool


Now we transform the multi-label narrative annotations into a **binary format suitable for machine learning** by removing the Other label, extracting all unique dominant and sub-narrative classes from the dataset, and encoding them using a multi-label binarization approach.

The resulting binary matrices are converted into DataFrames and concatenated with the original article metadata and text. Finally, a separate indicator column is added to explicitly mark articles labeled only as Other, producing the final structured dataset for modeling.

In [ ]:
#  helpers: strip 'Other' entries before binarizing
def without_other(label_list):
    return [l for l in label_list if l != 'Other']

# Build dom_mlb from labels actually present in the data, no label file needed
# Extract all dominant narrative labels
# Takes the column dom_narratives, Removes "Other" from every row, Collects all labels appearing in the dataset, Removes duplicates with set(), Sorts them alphabetically
all_dom_labels = sorted(set(
    l for labels in df_raw['dom_narratives'].apply(without_other)
    for l in labels
))

# scikit-learn MultiLabelBinarizer converts lists of labels into binary vectors. Convert labels into machine-readable binary vectors.
# Machine learning models require numeric targets.
dom_mlb = MultiLabelBinarizer(classes=all_dom_labels)
# Fit the binarizer
dom_mlb.fit([all_dom_labels])
# Transform labels into binary vectors
dom_bin = dom_mlb.transform(df_raw['dom_narratives'].apply(without_other))
# Convert to DataFrame
df_dom  = pd.DataFrame(dom_bin, columns=dom_mlb.classes_)

# same for sub
all_sub_labels = sorted(set(
    l for labels in df_raw['subnarratives'].apply(without_other)
    for l in labels
))
sub_mlb = MultiLabelBinarizer(classes=all_sub_labels)
sub_mlb.fit([all_sub_labels])
sub_bin = sub_mlb.transform(df_raw['subnarratives'].apply(without_other))
df_sub  = pd.DataFrame(sub_bin, columns=sub_mlb.classes_)

#  Rebuild df with the corrected binarization . This creates a dedicated binary column: 1 if the row contains only ['Other'], 0 otherwise
is_other = df_raw['dom_narratives'].apply(lambda ls: ls == ['Other']).astype(int)

# Build final dataset
df = pd.concat([df_raw[['article_id', 'language', 'text']], df_dom, df_sub], axis=1)
df['Other'] = is_other.values

print(df.shape)
df.head(2)

(399, 108)


,article_id,language,text,CC: Amplifying Climate Fears,CC: Climate change is beneficial,CC: Controversy about green technologies,CC: Criticism of climate movement,CC: Criticism of climate policies,CC: Criticism of institutions and authorities,CC: Downplaying climate change,...,URW: Praise of Russia: Russia is a guarantor of peace and prosperity,URW: Praise of Russia: Russian invasion has strong national support,URW: Russia is the Victim: Other,URW: Russia is the Victim: Russia actions in Ukraine are only self-defence,URW: Russia is the Victim: The West is russophobic,URW: Russia is the Victim: UA is anti-RU extremists,URW: Speculating war outcomes: Other,URW: Speculating war outcomes: Russian army is collapsing,URW: Speculating war outcomes: Ukrainian army is collapsing,Other
0,EN_CC_100013.txt,EN,Bill Gates Says He Is ‘The Solution’ To Climat...,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,EN_UA_300009.txt,EN,Russia: Clashes erupt in Bashkortostan as righ...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


---
# Subtask 2: Narrative Classification with Gensim Word2Vec

**Approach:**
- Pre-trained Google News Word2Vec (300-d) -> TF-IDF weighted document vectors
- Classifiers: Logistic Regression and SVM (both OneVsRest)
- Predict **sub-narratives (fine)** -> derive **dominant narratives (coarse)** from predictions
- Threshold: per-label tuned on validation fold
- Evaluation: 5-fold CV for model selection + final eval on gold test set
- F1@coarse and F1@fine reported


In [ ]:
# Install required packages
!pip install -q gensim nltk
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)  # lexical database used for lemmatization
nltk.download('omw-1.4',  quiet=True)   # multilingual WordNet support

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 66.9 MB/s eta 0:00:00


True

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import defaultdict
import gensim.downloader as api
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()
RANDOM_SEED = 42

##**Text Preprocessing**:

This code performs text preprocessing and normalization to prepare the articles for NLP modeling.

It defines a function that cleans raw text by converting it to lowercase, removing non-alphabetic characters, filtering out stopwords, and applying lemmatization to reduce words to their base forms. The result is a list of meaningful tokens suitable for models like embeddings or topic modeling.

A second utility function converts the token list back into a clean string format when needed.

Finally, the preprocessing is applied to the dataset, creating two new columns:
- tokens: tokenized and normalized word lists
- clean_text: reconstructed cleaned text as a string

In [ ]:
def preprocess(text: str) -> list[str]:
    """Lowercase -> strip non-alpha -> remove stopwords -> lemmatize.
    Returns a list of tokens (needed for Word2Vec lookup).
    """
    text  = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)  # substitute pattern by searching for patterns with \s = space, ^=not. Anything that matches the patter is substituted with ' '
    tokens = [LEMMATIZER.lemmatize(t) for t in text.split()
              if t not in STOP_WORDS and len(t) > 2]
    return tokens

def tokens_to_string(tokens: list[str]) -> str:     # words = ['From', 'tokens', 'to', 'string'], result = tokens_to_string(words) -> Output: 'From tokens to string'
    return ' '.join(tokens)   # concatenates tokens with ' ' as delimiter

df_raw = parse_annotations(TRAIN_ANNOTATIONS)
# Keep only EN
df_raw = df_raw[df_raw['language'] == LANG].reset_index(drop=True)
# Apply to training corpus
df['tokens'] = df['text'].fillna('').apply(preprocess)    # fillna replace any missing row in the text column with an empty string (''); you pass the outcome to pre process function
df['clean_text'] = df['tokens'].apply(tokens_to_string)
print(f"Sample tokens: {df['tokens'].iloc[0][:12]}")

Sample tokens: ['bill', 'gate', 'say', 'solution', 'climate', 'change', 'four', 'private', 'jet', 'bill', 'gate', 'right']


Load Pre-trained Word2Vec (Google News, 300-d)

In [ ]:
# Downloads around 1.6 GB the first time; cached afterwards
print('Loading Word2Vec model ...')
w2v_model = api.load('word2vec-google-news-300')
W2V_DIM   = 300
print(f'Loaded. Vocab size: {len(w2v_model.key_to_index):,}')

Loading Word2Vec model ...
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Loaded. Vocab size: 3,000,000


##**TF-IDF Weighted Document Vectors**:

It converts each document into a dense vector by:

- taking word embeddings from a trained Word2Vec model
- weighting each word by its TF-IDF importance
- averaging the weighted vectors

This gives a stronger representation than plain averaging because important words contribute more.

In [ ]:
def build_tfidf_w2v_matrix(clean_texts: list[str],
                            tokens_list: list[list[str]],
                            w2v,
                            tfidf_vectorizer: TfidfVectorizer | None = None
                            ) -> tuple[np.ndarray, TfidfVectorizer]:
    """
    Fit (or reuse) a TF-IDF vectorizer on `clean_texts`, then for each
    document compute a weighted average of Word2Vec token vectors using
    the TF-IDF scores as weights.

    Returns (X, fitted_tfidf_vectorizer).
    """
    # 1. Fit a new vectorizer if one isn't provided, otherwise use the existing one
    # The vectorizer is imprtant because measures word importance. (Common words receive low importance.)
    if tfidf_vectorizer is None:
        tfidf_vectorizer = TfidfVectorizer()
        tfidf_vectorizer.fit(clean_texts)

    # 2. Extract vocabulary and matching IDF values safely
    vocab = tfidf_vectorizer.vocabulary_
    idf = tfidf_vectorizer.idf_
    word2idf = {word: idf[idx] for word, idx in vocab.items()}

    # 3. Dynamically get the Word2Vec dimension size so it never errors out
    # Word embeddings map words into a continuous vector space where semantically similar words are close together,
    # related concepts share patterns and contextual meaning is partially preserved
    w2v_dim = w2v.vector_size if hasattr(w2v, 'vector_size') else len(next(iter(w2v.values())))

    # 4. Initialize embedding matrix
    X = np.zeros((len(tokens_list), w2v_dim), dtype=np.float32)

    # 5. Calculate weighted vector averages since Word2Vec alone treats all words equally. But not all words are equally informative. TF-IDF weights important words more strongly.
    for i, tokens in enumerate(tokens_list):    # loop for all documents
        vecs    = []
        weights = []
        for t in tokens:
            # Safely check if the token exists both in Word2Vec and the training TF-IDF vocab. If it’s a rare typo or an unknown symbol, Word2Vec won't have a vector for it.
            if t in w2v and t in word2idf:
                vecs.append(w2v[t])
                weights.append(word2idf[t])

        if vecs:
            weights = np.array(weights)
            X[i]    = np.average(vecs, axis=0, weights=weights)

    return X, tfidf_vectorizer

# Build full training matrix (used for final model training)
X_all, tfidf_vec = build_tfidf_w2v_matrix(
    df['clean_text'].tolist(),
    df['tokens'].tolist(),
    w2v_model
)
print(f'Document matrix shape: {X_all.shape}')
print(f"Embedding matrix first elements:\n{X_all[:2]}")
# Print the first 10 items in the vocabulary dictionary
first_10_vocab = dict(list(tfidf_vec.vocabulary_.items())[:10])
print(f"TF-IDF Vocabulary Samples:\n{first_10_vocab}")

Document matrix shape: (399, 300)
Embedding matrix first elements:
[[ 3.81175578e-02  3.04673780e-02  4.69153281e-03  7.77809024e-02
  -4.85036708e-02 -3.67444791e-02  6.24213694e-03 -9.69789103e-02
   1.16075277e-01  1.64271370e-02  2.08861157e-02 -1.29608765e-01
   3.50382691e-03 -1.50821349e-02 -1.19607590e-01  1.04160562e-01
   2.81432085e-02  7.45945275e-02  2.59580351e-02 -3.84068526e-02
  -1.95974391e-02  4.64016572e-02  1.05622672e-02 -6.75220392e-04
   5.46141015e-03 -2.52288743e-03 -7.58907348e-02  6.98531419e-02
  -2.56497366e-03 -3.89419571e-02 -1.44786099e-02 -7.63202831e-02
  -5.75703420e-02 -3.47086750e-02  2.63145585e-02 -6.07908368e-02
   1.83554261e-03  2.19914243e-02  7.71570951e-04  7.37041235e-02
   2.29219534e-02 -6.03098385e-02  1.00882411e-01 -1.04502663e-02
  -5.36089540e-02 -7.78429657e-02 -2.77471729e-02  3.69933546e-02
  -4.46664616e-02  2.13515703e-02  2.02689115e-02 -1.87189635e-02
   7.03318138e-03 -5.44377863e-02 -2.27264110e-02 -2.57877778e-04
  -8.2201

##Label Setup: Fine (sub) and Coarse (dominant):

Let's now prepare the label representations for the hierarchical classification setup.
We first encode both sub-narrative (fine-grained) and dominant narrative (coarse-grained) annotations into binary matrices, then build a mapping from sub-narratives to their corresponding dominant narratives so that fine-level predictions can later be aggregated into coarse-level predictions.

Since we are not doing a single prediction but two related prediction problems:
- Fine-grained task (harder) - predict sub-narratives (Y_sub) with 91 possible labels
- Coarse-grained task (easier) - predict dominant narratives (Y_dom) with 22 labels

So we are building a hierarchical label system.

The use of "*fine_pred_to_coarse(...)*" function is for evaluation. Let's make an example:

```
Sub-narratives:
- CC: Ice sheet melting
- CC: Extreme weather events

You want to check if the model at least get CC right at a higher level?”

So you convert: sub predictions -> dom predictions

There is also Hierarchical consistency: If a sub-label is CC, its parent must also be CC
```
If the model fails at fine level but succeeds at domain level: the model understands topic but not detail.

The actual goal, now, became to predict the SUB-narratives, and not the domain (we obtain the domain narratives by hierarchy).


In [ ]:
def without_other(label_list):
    return [l for l in label_list if l != 'Other']

#  Sub-narrative (fine) labels ─
all_sub_labels = sorted(set(
    l for labels in df_raw['subnarratives'].apply(without_other) for l in labels
))
sub_mlb = MultiLabelBinarizer(classes=all_sub_labels)
sub_mlb.fit([all_sub_labels])
Y_sub = sub_mlb.transform(df_raw['subnarratives'].apply(without_other))

#  Dominant (coarse) labels
all_dom_labels = sorted(set(
    l for labels in df_raw['dom_narratives'].apply(without_other) for l in labels
))
dom_mlb = MultiLabelBinarizer(classes=all_dom_labels)
dom_mlb.fit([all_dom_labels])
Y_dom = dom_mlb.transform(df_raw['dom_narratives'].apply(without_other))

#  Mapping: sub-narrative -> dominant narrative ─
def sub_to_dom(sub_label: str) -> str | None:
    """Extract dominant label from a sub-narrative string.
    e.g. 'CC: Criticism of climate movement: Ad hominem ...' -> 'CC: Criticism of climate movement'
    """
    for dom in all_dom_labels:
        if sub_label == dom or sub_label.startswith(dom + ':'):
            return dom
    return None

# Pre-build lookup table - a Python dictionary comprehension. Its purpose is to create a static lookup table (a dictionary) named sub2dom that maps every
# fine-grained category (sub-narrative) directly to its high-level parent category (dominant narrative).
sub2dom = {s: sub_to_dom(s) for s in all_sub_labels}

def fine_pred_to_coarse(Y_sub_pred: np.ndarray) -> np.ndarray:
    """Convert a binary sub-narrative prediction matrix to a dominant-narrative matrix.
    It translates an entire block of multi-label binary predictions from the fine level to the coarse level."""
    Y_dom_pred = np.zeros((Y_sub_pred.shape[0], len(all_dom_labels)), dtype=int)    # The shape is (number of documents, number of dominant labels).
    for sub_idx, sub_lbl in enumerate(sub_mlb.classes_):    # The code loops through every sub-narrative category tracked by the MultiLabelBinarizer
        dom_lbl = sub2dom.get(sub_lbl)                      # It looks up what dom_lbl belongs to the current sub_lbl using our pre-built lookup table.
        if dom_lbl and dom_lbl in dom_mlb.classes_:
            dom_idx = list(dom_mlb.classes_).index(dom_lbl)   # It locates the exact column index (dom_idx) where that dominant label lives in the target matrix.
            Y_dom_pred[:, dom_idx] |= Y_sub_pred[:, sub_idx]  # The Bitwise OR Pooling (|=): If a document triggers ANY sub-narrative belonging to this dominant narrative, mark the dominant narrative as a 1. If none are triggered, leave it as 0.
    return Y_dom_pred

print(f'Fine labels  (sub) : {len(all_sub_labels)}')
print(f'Coarse labels (dom): {len(all_dom_labels)}')
print(f'Y_sub shape        : {Y_sub.shape}')
print(f'Y_dom shape        : {Y_dom.shape}')

Fine labels  (sub) : 83
Coarse labels (dom): 21
Y_sub shape        : (399, 83)
Y_dom shape        : (399, 21)


##Threshold tuning for multi-label classification

In multi-label classification, the model does not output a single class, but a probability for each label independently. Instead of using a fixed threshold (e.g., 0.5) for all labels, we optimize a separate threshold for each label to improve performance.

The function *tune_thresholds* searches, for every label, the threshold in a given range that *maximizes the binary F1-score on the validation set*. This allows the model to adapt to class imbalance and different confidence distributions across labels.

Once optimal thresholds are found, *apply_thresholds* converts predicted probabilities into final binary predictions by applying these learned per-label thresholds.

This step is important because it significantly improves multi-label F1 performance compared to using a single global cutoff.

In [ ]:
def tune_thresholds(y_true: np.ndarray,
                    y_proba: np.ndarray,
                    thresholds: np.ndarray = np.arange(0.1, 0.91, 0.05)
                    ) -> np.ndarray:
    # np.arange(0.1, 0.91, 0.05) = from 0.1 to 0.91 with a step of 0.05 (0.10, 0.15, 0.20, \dots, 0.90)
    """
    For each label independently, find the threshold in 'thresholds'
    that maximises binary F1 on (y_true, y_proba).
    Returns an array of shape (n_labels,) with the best threshold per label.
    """
    n_labels = y_true.shape[1]
    best_thresholds = np.full(n_labels, 0.5) # Initialize all thresholds to 0.5
    for j in range(n_labels):   # it loops for categories
        best_f1 = -1.0
        for t in thresholds:
            preds = (y_proba[:, j] >= t).astype(int)
            f1    = f1_score(y_true[:, j], preds, zero_division=0)  # zero_division=0: Returns 0.0 when undefined. A model that predicts nothing useful should get an F1 score of 0.
            if f1 > best_f1:
                best_f1           = f1
                best_thresholds[j] = t
    return best_thresholds

# Once you've used your validation data to learn the ideal thresholds, you need to apply them to your test set.
def apply_thresholds(y_proba: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    return (y_proba >= thresholds).astype(int)

## Evaluation Helpers (F1@fine, F1@coarse)

The *evaluate function* computes F1-score (micro-averaged) for both:
- fine-grained predictions (sub-narratives)
- coarse-grained predictions (dominant narratives)

Micro-F1 is used because it accounts for class imbalance.


Different classifiers in scikit-learn do not output probabilities in the same way To unify them, it was defined the '*get_proba*' function:

- Case 1: probabilistic models
- Case 2: non-probabilistic models (e.g. LinearSVC)

We convert decision scores into pseudo-probabilities using a sigmoid function:

This ensures outputs are always in the range [0, 1], making them compatible with threshold tuning.

In [ ]:
def evaluate(y_true_sub, y_pred_sub, y_true_dom, y_pred_dom, label=''):
    f1_fine   = f1_score(y_true_sub, y_pred_sub, average='micro', zero_division=0)
    f1_coarse = f1_score(y_true_dom, y_pred_dom, average='micro', zero_division=0)
    if label:
        print(f'[{label}]  F1@fine={f1_fine:.4f}   F1@coarse={f1_coarse:.4f}')
    return f1_fine, f1_coarse

def get_proba(classifier, X):
    """Return probability estimates; fall back to decision_function for LinearSVC."""
    if hasattr(classifier, 'predict_proba'):
        return classifier.predict_proba(X)
    scores = classifier.decision_function(X)
    # sigmoid normalisation per label
    return 1 / (1 + np.exp(-scores))

## 5-Fold Cross-Validation (Model Selection)

This block implements a full machine learning evaluation loop to fairly compare models for multi-label narrative classification.

The goal is to estimate how well different classifiers (Logistic Regression and SVM) generalize to unseen data while avoiding data leakage.

Two classifiers are tested:
- Logistic Regression (LR)
- Linear SVM (SVM)

Both are wrapped in: scikit-learn OneVsRestClassifier because this is a multi-label problem, where each label is treated as an independent binary task.

In [ ]:
classifiers = {
    'LR' : OneVsRestClassifier(LogisticRegression(max_iter=1000, C=1.0,
                                                   random_state=RANDOM_SEED)),
    'SVM': OneVsRestClassifier(LinearSVC(max_iter=2000, C=1.0,
                                          random_state=RANDOM_SEED)),
}

# Set up 5-fold cross-validation setup to split the data randomly but reproducibly
N_FOLDS = 5
kf      = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# Storage dictionary to automatically track lists of macro F1 scores for each model
# If you try to access or write to a key that doesn't exist yet, the defaultdict doesn't crash. Instead, it uses the lueprint to create the key (useful for dinamicity)
# the blueprint is given by lambda (anonymous function)
cv_results = defaultdict(lambda: {'f1_fine': [], 'f1_coarse': []})

# Loop through each of the 5 folds. 'enumerate(..., 1)' keeps track of fold count starting at 1.
for fold, (train_idx, val_idx) in enumerate(kf.split(X_all), 1):
    print(f"\n--- Processing Fold {fold}/{N_FOLDS} ---")

    # Split features into training and validation sets for this specific slice
    X_tr, X_val   = X_all[train_idx], X_all[val_idx]
    # Split the target matrices (Ys = Sub-narratives / fine, Yd = Dominant / coarse)
    Ys_tr, Ys_val = Y_sub[train_idx], Y_sub[val_idx]
    Yd_val        = Y_dom[val_idx]

    # Extract raw text data for this specific fold to perform localized vectorization
    clean_tr  = [df['clean_text'].iloc[i] for i in train_idx]
    tokens_tr = [df['tokens'].iloc[i]     for i in train_idx]
    tokens_val= [df['tokens'].iloc[i]     for i in val_idx]

    # TF-IDF is re-trained ONLY on training data of that fold to avoid data-leackage (TF-IDF is re-trained ONLY on training data of that fold)
    X_tr_fold, tfidf_fold = build_tfidf_w2v_matrix(clean_tr, tokens_tr, w2v_model)
    # Pass the fitted vectorizer into the validation set. This prevents the validation data
    # from influencing TF-IDF vocab/weights (strictly avoiding data leakage).
    X_val_fold, _         = build_tfidf_w2v_matrix([], tokens_val, w2v_model,
                                                     tfidf_vectorizer=tfidf_fold)

    # Model Training
    for name, clf in classifiers.items():
        clf.fit(X_tr_fold, Ys_tr)
        # Generate raw prediction probabilities for the validation set
        proba     = get_proba(clf, X_val_fold)
        # Dynamic Threshold Tuning: Find the best decision threshold for each separate label based entirely on the training split performance
        thresholds = tune_thresholds(Ys_tr, get_proba(clf, X_tr_fold))  # tune on train fold
        pred_sub  = apply_thresholds(proba, thresholds)
        # Map probabilities to binary (0 or 1) using the optimized thresholds
        pred_dom  = fine_pred_to_coarse(pred_sub)
        f1f, f1c  = evaluate(Ys_val, pred_sub, Yd_val, pred_dom,
                              label=f'Fold {fold} | {name}')
        cv_results[name]['f1_fine'].append(f1f)
        cv_results[name]['f1_coarse'].append(f1c)

print()
print('=== Cross-Validation Summary ===')
for name, res in cv_results.items():
    print(f'{name}:  '
          f'F1@fine={np.mean(res["f1_fine"]):.4f} +/- {np.std(res["f1_fine"]):.4f}  |  '
          f'F1@coarse={np.mean(res["f1_coarse"]):.4f} +/- {np.std(res["f1_coarse"]):.4f}')

import json
print("Macro Values: \n", json.dumps(dict(cv_results), indent=4))


--- Processing Fold 1/5 ---


/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 30 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 71 is present in all training examples.
  warnings.warn(


[Fold 1 | LR]  F1@fine=0.1206   F1@coarse=0.1818


/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 30 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 71 is present in all training examples.
  warnings.warn(


[Fold 1 | SVM]  F1@fine=0.1037   F1@coarse=0.1711

--- Processing Fold 2/5 ---


...

[Fold 3 | SVM]  F1@fine=0.1151   F1@coarse=0.2216

--- Processing Fold 4/5 ---
[Fold 4 | LR]  F1@fine=0.1263   F1@coarse=0.2029
[Fold 4 | SVM]  F1@fine=0.2007   F1@coarse=0.3756

--- Processing Fold 5/5 ---


/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 1 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 2 is present in all training examples.
  warnings.warn(


[Fold 5 | LR]  F1@fine=0.0796   F1@coarse=0.1569


/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 1 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 2 is present in all training examples.
  warnings.warn(


[Fold 5 | SVM]  F1@fine=0.1686   F1@coarse=0.2953

=== Cross-Validation Summary ===
LR:  F1@fine=0.0991 +/- 0.0240  |  F1@coarse=0.1657 +/- 0.0281
SVM:  F1@fine=0.1490 +/- 0.0356  |  F1@coarse=0.2547 +/- 0.0726
Macro Values: 
 {
    "LR": {
        "f1_fine": [
            0.12060301507537688,
            0.10526315789473684,
            0.06349206349206349,
            0.12631578947368421,
            0.07960199004975124
        ],
        "f1_coarse": [
            0.18181818181818182,
            0.16822429906542055,
            0.11851851851851852,
            0.2028985507246377,
            0.1568627450980392
        ]
    },
    "SVM": {
        "f1_fine": [
            0.1037037037037037,
            0.1568627450980392,
            0.11510791366906475,
            0.20074349442379183,
            0.16856492027334852
        ],
        "f1_coarse": [
            0.1710914454277286,
            0.2099125364431487,
            0.2215909090909091,
            0.3756345177664975,
   

## Train Final Model on Full Training Set

In [ ]:
# Choose best classifier based on CV F1@fine
best_clf_name = max(cv_results, key=lambda n: np.mean(cv_results[n]['f1_fine']))
print(f'Best classifier: {best_clf_name}')

final_clf = classifiers[best_clf_name]
final_clf.fit(X_all, Y_sub)

# Tune thresholds on the full training set
proba_train    = get_proba(final_clf, X_all)
final_thresholds = tune_thresholds(Y_sub, proba_train)
print('Final per-label thresholds (first 10):', np.round(final_thresholds[:10], 2))

Best classifier: SVM
Final per-label thresholds (first 10): [0.4  0.3  0.35 0.3  0.3  0.35 0.35 0.35 0.35 0.35]


## Final Evaluation on Gold Test Set

In [ ]:
# Load & preprocess test documents
df_test = parse_annotations(GOLD_FILE)
df_test = df_test[df_test['language'] == LANG].reset_index(drop=True)
test_texts = load_texts(TEST_DOCS_DIR, df_test['article_id'].tolist())
df_test['text']       = df_test['article_id'].map(test_texts)
df_test['tokens']     = df_test['text'].fillna('').apply(preprocess)
df_test['clean_text'] = df_test['tokens'].apply(tokens_to_string)

# Vectorise with the TF-IDF fitted on full training set
X_test, _ = build_tfidf_w2v_matrix(
    [], df_test['tokens'].tolist(), w2v_model, tfidf_vectorizer=tfidf_vec
)

# Gold labels
Y_test_sub = sub_mlb.transform(df_test['subnarratives'].apply(without_other))
Y_test_dom = dom_mlb.transform(df_test['dom_narratives'].apply(without_other))

# Predict
proba_test   = get_proba(final_clf, X_test)
pred_sub_test = apply_thresholds(proba_test, final_thresholds)
pred_dom_test = fine_pred_to_coarse(pred_sub_test)

print('=== FINAL TEST SET RESULTS ===')
f1f, f1c = evaluate(Y_test_sub, pred_sub_test, Y_test_dom, pred_dom_test,
                    label=f'{best_clf_name} on test')
print()
# Per-label breakdown
print('--- F1 per sub-narrative (fine) ---')
per_label_f1 = f1_score(Y_test_sub, pred_sub_test, average=None, zero_division=0)
for lbl, score in sorted(zip(sub_mlb.classes_, per_label_f1), key=lambda x: -x[1]):
    print(f'  {score:.3f}  {lbl}')

=== FINAL TEST SET RESULTS ===
[SVM on test]  F1@fine=0.2278   F1@coarse=0.3636

--- F1 per sub-narrative (fine) ---
  1.000  URW: Discrediting Ukraine: Ukraine is a hub for criminal activities
  1.000  URW: Negative Consequences for the West: Sanctions imposed by Western countries will backfire
  0.857  CC: Criticism of climate movement: Other
  0.800  CC: Questioning the measurements and science: Methodologies/metrics used are unreliable/faulty
  0.667  URW: Blaming the war on others rather than the invader: The West are the aggressors
  0.500  CC: Controversy about green technologies: Renewable energy is costly
  0.500  CC: Criticism of climate movement: Climate movement is alarmist
  0.400  CC: Criticism of climate policies: Climate policies have negative impact on the economy
  0.400  CC: Criticism of institutions and authorities: Criticism of national governments
  0.364  URW: Discrediting the West, Diplomacy: Other
  0.250  CC: Criticism of institutions and authorities: Criticis

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['CC: Green policies are geopolitical instruments: Other', 'URW: Discrediting Ukraine: Discrediting Ukrainian nation and society', 'URW: Discrediting Ukraine: Other'] will be ignored
  warnings.warn(


## Save Predictions to Output File

In [ ]:
with open(OUTPUT_FILE, 'w', encoding='utf-8') as fout:
    for i, row in df_test.iterrows():
        pred_subs = [sub_mlb.classes_[j]
                     for j in range(len(sub_mlb.classes_))
                     if pred_sub_test[i, j] == 1]
        pred_doms = list(dict.fromkeys(
            sub2dom[s] for s in pred_subs if sub2dom.get(s)
        ))
        if not pred_subs:
            pred_subs = ['Other']
            pred_doms = ['Other']
        dom_str = ';'.join(pred_doms)
        sub_str = ';'.join(pred_subs)
        fout.write(f"{row['article_id']}\t{dom_str}\t{sub_str}\n")

print(f'Predictions saved to {OUTPUT_FILE}')

Predictions saved to /content/drive/MyDrive/NLP_Project/predictions_subtask2_EN.txt



---

Let's try to improve the results by apporting some modifications to what we've done till now

## Improved CV with C grid search

In this step, we perform a systematic search to find the best value of the regularization parameter C for the Linear SVM classifier. The parameter C controls the trade-off between model complexity and generalization: smaller values produce simpler models with stronger regularization, while larger values allow the model to fit the training data more closely.

In [ ]:
#  C Grid Search CV (no balanced, original threshold tuning)
from sklearn.model_selection import KFold

C_VALUES = [0.1, 0.5, 1.0, 5.0, 10.0]
N_FOLDS  = 5
kf       = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

cv_results_c = {c: {'f1_fine': [], 'f1_coarse': []} for c in C_VALUES}

for fold, (train_idx, val_idx) in enumerate(kf.split(X_all), 1):
    X_tr,  X_val  = X_all[train_idx], X_all[val_idx]
    Ys_tr, Ys_val = Y_sub[train_idx], Y_sub[val_idx]
    Yd_val        = Y_dom[val_idx]

    clean_tr   = [df['clean_text'].iloc[i] for i in train_idx]
    tokens_tr  = [df['tokens'].iloc[i]     for i in train_idx]
    tokens_val = [df['tokens'].iloc[i]     for i in val_idx]

    X_tr_f, tfidf_f = build_tfidf_w2v_matrix(clean_tr, tokens_tr, w2v_model)
    X_val_f, _      = build_tfidf_w2v_matrix([], tokens_val, w2v_model,
                                               tfidf_vectorizer=tfidf_f)
    for C in C_VALUES:
        clf = OneVsRestClassifier(
            LinearSVC(C=C, max_iter=3000, random_state=RANDOM_SEED)
        )
        clf.fit(X_tr_f, Ys_tr)

        # tune thresholds on training fold (same as original working pipeline)
        thresholds = tune_thresholds(Ys_tr, get_proba(clf, X_tr_f))

        pred_sub = apply_thresholds(get_proba(clf, X_val_f), thresholds)
        pred_dom = fine_pred_to_coarse(pred_sub)
        f1f, f1c = evaluate(Ys_val, pred_sub, Yd_val, pred_dom,
                            label=f'Fold {fold} | C={C}')
        cv_results_c[C]['f1_fine'].append(f1f)
        cv_results_c[C]['f1_coarse'].append(f1c)

print()
print(f'{"C":>6}  {"F1@fine":>10}  {"+/-":>6}  {"F1@coarse":>10}  {"+/-":>6}')
print('-' * 50)
for C in C_VALUES:
    r = cv_results_c[C]
    print(f'{C:>6}  {np.mean(r["f1_fine"]):>10.4f}  '
          f'{np.std(r["f1_fine"]):>6.4f}  '
          f'{np.mean(r["f1_coarse"]):>10.4f}  '
          f'{np.std(r["f1_coarse"]):>6.4f}')

/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 30 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 71 is present in all training examples.
  warnings.warn(


[Fold 1 | C=0.1]  F1@fine=0.0435   F1@coarse=0.1170


/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 30 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 71 is present in all training examples.
  warnings.warn(


[Fold 1 | C=0.5]  F1@fine=0.1016   F1@coarse=0.2194


/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 30 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 71 is present in all training examples.
  warnings.warn(


...

[Fold 5 | C=1.0]  F1@fine=0.1686   F1@coarse=0.2953


/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 1 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 2 is present in all training examples.
  warnings.warn(


[Fold 5 | C=5.0]  F1@fine=0.1810   F1@coarse=0.2877


/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 1 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 2 is present in all training examples.
  warnings.warn(


[Fold 5 | C=10.0]  F1@fine=0.1980   F1@coarse=0.3086

     C     F1@fine       ±   F1@coarse       ±
--------------------------------------------------
   0.1      0.0466  0.0041      0.1240  0.0143
   0.5      0.1015  0.0146      0.1981  0.0259
   1.0      0.1490  0.0356      0.2547  0.0726
   5.0      0.1604  0.0304      0.2652  0.0611
  10.0      0.1659  0.0311      0.2786  0.0498


Retrain best config and evaluate on test:

In [ ]:
#  Retrain best C and evaluate on test ─
best_C = max(C_VALUES, key=lambda c: np.mean(cv_results_c[c]['f1_fine']))
print(f'Best C: {best_C}')

final_clf_c = OneVsRestClassifier(
    LinearSVC(C=best_C, max_iter=3000, random_state=RANDOM_SEED)
)
final_clf_c.fit(X_all, Y_sub)

proba_train    = get_proba(final_clf_c, X_all)
final_thresh_c = tune_thresholds(Y_sub, proba_train)

proba_test_c = get_proba(final_clf_c, X_test)
pred_sub_c   = apply_thresholds(proba_test_c, final_thresh_c)
pred_dom_c   = fine_pred_to_coarse(pred_sub_c)

print(f'Avg labels per doc: {pred_sub_c.sum(axis=1).mean():.2f}')
print()
print('=== C Grid Search — TEST RESULTS ===')
evaluate(Y_test_sub, pred_sub_c, Y_test_dom, pred_dom_c, label=f'SVM C={best_C}')
print()
print('--- Comparison ---')
print(f'Original (C=1.0):  '
      f'F1@fine={f1_score(Y_test_sub, pred_sub_test,  average="micro", zero_division=0):.4f}  '
      f'F1@coarse={f1_score(Y_test_dom, pred_dom_test, average="micro", zero_division=0):.4f}')

Best C: 10.0
Avg labels per doc: 2.32

=== C Grid Search — TEST RESULTS ===
[SVM C=10.0]  F1@fine=0.2974   F1@coarse=0.4133

--- Comparison ---
Original (C=1.0):  F1@fine=0.2278  F1@coarse=0.3636


Save the output file

In [ ]:
#  Save best predictions
OUTPUT_FILE_BEST = f'{BASE}/predictions_subtask2_EN_v2.txt'

with open(OUTPUT_FILE_BEST, 'w', encoding='utf-8') as fout:
    for i, row in df_test.iterrows():
        pred_subs = [sub_mlb.classes_[j]
                     for j in range(len(sub_mlb.classes_))
                     if pred_sub_c[i, j] == 1]
        pred_doms = list(dict.fromkeys(
            sub2dom[s] for s in pred_subs if sub2dom.get(s)
        ))
        if not pred_subs:
            pred_subs = ['Other']
            pred_doms = ['Other']
        dom_str = ';'.join(pred_doms)
        sub_str = ';'.join(pred_subs)
        fout.write(f"{row['article_id']}\t{dom_str}\t{sub_str}\n")

print(f'Best predictions saved to {OUTPUT_FILE_BEST}')

Best predictions saved to /content/drive/MyDrive/NLP_Project/predictions_subtask2_EN_v2.txt




> ***NLP Case Study - Subtask_2 | Manuel Carlucci | Mtr: 855237 | m.carlucci69@studenti.uniba.it***

